[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/duckdb-certified/notebooks/day-05-pandas-polars-arrow.ipynb#scrollTo=10a2b3c4)

---
# Day 5 · Zero-Copy Integration — Pandas, Polars, and Apache Arrow
**certified-journeys / duckdb-certified** · Practice · DataFrame Integrations

> **Goal for today:** By the end of this notebook you can query Pandas and Polars DataFrames directly with SQL (no serialisation), exchange data via Arrow's zero-copy interface, and understand when DuckDB is faster than native DataFrame operations.


In [ ]:
%pip install -q duckdb pandas polars pyarrow


## Step 1 · Querying a Pandas DataFrame directly with SQL

DuckDB scans the Pandas DataFrame **in-place** — it reads the underlying NumPy or Arrow memory buffer rather than copying data.
You just use the DataFrame's variable name directly in SQL.

Reference: [DuckDB + Pandas Guide](https://duckdb.org/docs/guides/python/sql_on_pandas)

| Approach | Copy? | Memory overhead |
|----------|-------|-----------------|
| `duckdb.sql('SELECT … FROM df')` | No | ~0 |
| `pd.DataFrame.merge()` | Yes | Full copy |
| `pandas.read_sql()` → push to DuckDB | Yes | Full copy |


In [ ]:
import duckdb
import pandas as pd
import numpy as np

# Create a Pandas DataFrame — DuckDB will reference this by variable name
rng = np.random.default_rng(42)
df_sales = pd.DataFrame({
    'order_id':  range(1, 101),
    'customer':  rng.choice(['Alice', 'Bob', 'Carol', 'Dave', 'Eve'], 100),
    'product':   rng.choice(['Widget', 'Gadget', 'Doohickey'], 100),
    'amount':    rng.uniform(10, 500, 100).round(2),
    'region':    rng.choice(['North', 'South', 'East', 'West'], 100),
})

# Query the DataFrame directly — no copy, no register() call needed
top_customers = duckdb.sql("""
    SELECT customer,
           COUNT(*)             AS orders,
           ROUND(SUM(amount), 2) AS total_spent
    FROM df_sales
    GROUP BY customer
    ORDER BY total_spent DESC
""").df()

print("Top customers by spend:")
print(top_customers)

# Filter with a WHERE clause — SQL pushdown works on the Pandas buffer
north_sales = duckdb.sql("""
    SELECT product,
           ROUND(AVG(amount), 2) AS avg_amount,
           COUNT(*)               AS cnt
    FROM df_sales
    WHERE region = 'North' AND amount > 100
    GROUP BY product
    ORDER BY avg_amount DESC
""").df()

print("\nNorth region (amount > 100):")
print(north_sales)


### What just happened?
- **`FROM df_sales`** references the Python variable by name — DuckDB's Python scanner resolves it automatically in the current scope.
- No `con.register()` call was needed here because `duckdb.sql()` searches the caller's local namespace.
- **Predicate pushdown** (`WHERE region = 'North'`) filters rows before aggregation — the planner does not materialise the full DataFrame in DuckDB.
- **`.df()`** at the end of a `duckdb.sql()` call converts the result set back into a Pandas DataFrame.


## Step 2 · Converting DuckDB results back to Pandas with `.df()`

Every DuckDB result relation exposes `.df()` (alias: `.fetchdf()`). This is the primary bridge back into the Pandas ecosystem.

**Typical round-trip pattern:**
```
df_raw  → duckdb.sql(SQL) → DuckDBPyRelation → .df() → df_result
```

Type mapping to remember:

| DuckDB type | Pandas dtype |
|-------------|-------------|
| `INTEGER` | `int64` |
| `DOUBLE` | `float64` |
| `VARCHAR` | `object` |
| `DATE` | `datetime64[ns]` |
| `TIMESTAMP` | `datetime64[ns]` |
| `BOOLEAN` | `bool` |
| `LIST` | `object` (list) |


In [ ]:
import duckdb
import pandas as pd

# Start from a Pandas DataFrame
df_events = pd.DataFrame({
    'event_id':   range(1, 201),
    'user_id':    [f'u{i % 20 + 1:03d}' for i in range(200)],
    'event_type': ['page_view' if i % 3 != 0 else 'purchase' for i in range(200)],
    'amount':     [round((i % 10 + 1) * 9.99, 2) if i % 3 == 0 else None for i in range(200)],
})

# SQL aggregation over Pandas DataFrame → back to Pandas
purchase_summary = duckdb.sql("""
    SELECT
        user_id,
        COUNT(*)          FILTER (event_type = 'purchase') AS purchases,
        COUNT(*)          FILTER (event_type = 'page_view') AS page_views,
        COALESCE(SUM(amount), 0)                           AS total_spend
    FROM df_events
    GROUP BY user_id
    HAVING purchases > 0
    ORDER BY total_spend DESC
    LIMIT 5
""").df()

print("Top 5 purchasing users:")
print(purchase_summary)
print("\nResult dtypes:")
print(purchase_summary.dtypes)

# Window function — running total per user (impossible as a simple pandas groupby)
running_total = duckdb.sql("""
    SELECT
        event_id,
        user_id,
        amount,
        SUM(amount) OVER (PARTITION BY user_id ORDER BY event_id) AS running_spend
    FROM df_events
    WHERE event_type = 'purchase' AND user_id = 'u001'
""").df()
print("\nRunning spend for u001:")
print(running_total)


### What just happened?
- **`FILTER (event_type = 'purchase')`** is a DuckDB-native conditional aggregate — cleaner than a pandas `.pivot_table()` approach.
- **`HAVING purchases > 0`** filters groups after aggregation — this runs entirely in DuckDB before the result reaches Python.
- **Window functions** (`SUM … OVER PARTITION BY`) are far more readable in SQL than the `groupby().cumsum()` Pandas equivalent.
- **`.df()` preserves NULL** as `NaN` in Pandas `float64` columns — `COALESCE` upstream prevents nulls leaking into downstream code.


## Step 3 · Polars integration — querying LazyFrames

Polars is a Rust-backed DataFrame library with lazy evaluation. DuckDB can query both Polars `DataFrame` and `LazyFrame` objects directly.

Reference: [DuckDB + Polars Guide](https://duckdb.org/docs/guides/python/polars)

| Polars type | DuckDB support | Notes |
|-------------|---------------|-------|
| `polars.DataFrame` | Full read via Arrow IPC | Zero-copy |
| `polars.LazyFrame` | Full read (materialises first) | DuckDB triggers `.collect()` |

Polars uses **Arrow-native** columnar storage, so DuckDB reads it via the same zero-copy Arrow path as `pyarrow.Table`.


In [ ]:
import duckdb
import polars as pl
import numpy as np

rng = np.random.default_rng(7)

# --- Polars DataFrame ---
pl_products = pl.DataFrame({
    'product_id':  list(range(1, 51)),
    'category':    rng.choice(['A', 'B', 'C'], 50).tolist(),
    'unit_cost':   rng.uniform(1, 100, 50).round(2).tolist(),
    'units_sold':  rng.integers(0, 500, 50).tolist(),
})

# Query the Polars DataFrame directly from DuckDB
result_df = duckdb.sql("""
    SELECT category,
           COUNT(*)               AS products,
           ROUND(SUM(unit_cost * units_sold), 2) AS revenue,
           ROUND(AVG(unit_cost), 2)              AS avg_cost
    FROM pl_products
    GROUP BY category
    ORDER BY revenue DESC
""").df()
print("Category revenue (from Polars DataFrame):")
print(result_df)

# --- Polars LazyFrame (scan_csv equivalent) ---
# DuckDB will call .collect() internally before scanning
pl_lazy = pl_products.lazy().filter(pl.col('units_sold') > 100)

# Must collect first when using implicit scope scanning
# (DuckDB triggers .collect() for LazyFrames automatically in newer builds,
#  but explicit collect is safer across versions)
pl_collected = pl_lazy.collect()

top_sellers = duckdb.sql("""
    SELECT product_id, category, unit_cost, units_sold,
           ROUND(unit_cost * units_sold, 2) AS revenue
    FROM pl_collected
    ORDER BY revenue DESC
    LIMIT 5
""").df()
print("\nTop 5 sellers (units_sold > 100):")
print(top_sellers)

# --- Result back to Polars ---
pl_result = duckdb.sql("""
    SELECT category, MAX(units_sold) AS max_units
    FROM pl_products
    GROUP BY category
""").pl()   # .pl() returns a Polars DataFrame directly
print("\nResult as Polars DataFrame:")
print(pl_result)


### What just happened?
- **`FROM pl_products`** works because DuckDB finds the Polars DataFrame in the caller's local scope and reads it via the **Arrow IPC** interface — no Python-level copy.
- **`.pl()`** on a DuckDB result returns a `polars.DataFrame` — the complement of `.df()` for the Polars ecosystem.
- LazyFrames need `.collect()` before passing them as a SQL source in most DuckDB versions — always collect for reliability.
- Polars' own aggregation engine is fast, but **DuckDB's SQL optimizer** often wins on multi-join or window-heavy queries by using better physical plans.


## Step 4 · Apache Arrow — zero-copy via `.arrow()` and `.fetch_arrow_table()`

Apache Arrow defines a columnar in-memory format that DuckDB, Pandas (via `pandas>=1.5`), Polars, and many other tools share natively.
Passing data through Arrow avoids serialisation entirely — the same memory buffer is read by each library.

Reference: [DuckDB + Apache Arrow](https://duckdb.org/docs/guides/python/sql_on_arrow)

```
pyarrow.Table  ──→  DuckDB SQL  ──→  .arrow()  ──→  pyarrow.Table
                                     .pl()     ──→  polars.DataFrame
                                     .df()     ──→  pandas.DataFrame
```


In [ ]:
import duckdb
import pyarrow as pa
import pyarrow.compute as pc
import pandas as pd

# Build a PyArrow table
arrow_tbl = pa.table({
    'city':       ['NYC', 'LA', 'Chicago', 'Houston', 'Phoenix',
                   'Philadelphia', 'San Antonio', 'San Diego', 'Dallas', 'San Jose'],
    'state':      ['NY', 'CA', 'IL', 'TX', 'AZ', 'PA', 'TX', 'CA', 'TX', 'CA'],
    'population': [8336817, 3979576, 2693976, 2304580, 1608139,
                   1603797, 1434625, 1386932, 1304379, 1013240],
    'area_sqmi':  [302.6, 503.0, 227.7, 669.0, 517.9,
                   142.0, 460.9, 372.1, 385.8, 177.5],
})

# Query the Arrow table directly
density_result = duckdb.sql("""
    SELECT city,
           state,
           population,
           ROUND(population / area_sqmi, 0) AS pop_density
    FROM arrow_tbl
    ORDER BY pop_density DESC
""").arrow()   # return as another Arrow table

print("Result type:", type(density_result))
print("Schema:", density_result.schema)
print("Top 3 densest cities:")
for i in range(3):
    row = {col: density_result.column(col)[i].as_py() for col in density_result.schema.names}
    print(f"  {row}")

# Arrow → Pandas via zero-copy
df_from_arrow = density_result.to_pandas()
print("\nAs Pandas DataFrame:")
print(df_from_arrow)

# State-level aggregation — result straight to Arrow
state_agg = duckdb.sql("""
    SELECT state,
           COUNT(*)             AS cities,
           SUM(population)      AS total_pop,
           ROUND(AVG(area_sqmi), 1) AS avg_area
    FROM arrow_tbl
    GROUP BY state
    ORDER BY total_pop DESC
""").fetch_arrow_table()   # alias for .arrow()

print("\nState summary (Arrow table):")
print(state_agg.to_pandas())


### What just happened?
- **`FROM arrow_tbl`** references a `pyarrow.Table` in scope — DuckDB reads the Arrow buffer pointers directly, no copy.
- **`.arrow()`** and **`.fetch_arrow_table()`** are identical — both return a `pyarrow.Table`.
- **`arrow_tbl.to_pandas()`** is a zero-copy conversion when the Arrow types map natively (integers, floats, strings with `zero_copy_only=True`); strings may copy in older pyarrow versions.
- **Arrow as interchange format** lets you move between DuckDB, Pandas, Polars, and Parquet without ever leaving the columnar memory model.


## Step 5 · Joining a DuckDB table with a Pandas DataFrame

One of DuckDB's most powerful patterns is **mixing storage backends in a single SQL query** — joining a persistent DuckDB table with an in-memory Pandas DataFrame as if they were the same thing.

This works because DuckDB's query planner treats registered Python objects and SQL-named variables as equal citizens.


In [ ]:
import duckdb
import pandas as pd

con = duckdb.connect()

# Persistent DuckDB table — imagine this is on disk
con.execute("""
    CREATE TABLE transactions AS
    SELECT
        i           AS txn_id,
        'cust_' || LPAD(CAST((i % 5 + 1) AS VARCHAR), 3, '0') AS customer_id,
        ROUND(50 + (i * 7.3 % 450), 2)          AS amount,
        DATE '2024-01-01' + (i % 90)             AS txn_date
    FROM range(1, 31) t(i)
""")

# In-memory Pandas DataFrame — e.g., enrichment data from an API
df_customers = pd.DataFrame({
    'customer_id': [f'cust_{i:03d}' for i in range(1, 6)],
    'name':        ['Alice', 'Bob', 'Carol', 'Dave', 'Eve'],
    'tier':        ['Gold', 'Silver', 'Gold', 'Bronze', 'Silver'],
})

# JOIN persistent table with Pandas DataFrame in one query
enriched = con.execute("""
    SELECT
        t.txn_id,
        c.name,
        c.tier,
        t.amount,
        t.txn_date
    FROM transactions t
    JOIN df_customers c ON t.customer_id = c.customer_id
    ORDER BY t.amount DESC
    LIMIT 8
""").fetchdf()
print("Enriched transactions:")
print(enriched)

# Aggregate across the join
tier_summary = con.execute("""
    SELECT c.tier,
           COUNT(t.txn_id)           AS txn_count,
           ROUND(SUM(t.amount), 2)   AS total_spend,
           ROUND(AVG(t.amount), 2)   AS avg_txn
    FROM transactions t
    JOIN df_customers c ON t.customer_id = c.customer_id
    GROUP BY c.tier
    ORDER BY total_spend DESC
""").fetchdf()
print("\nSpend by customer tier:")
print(tier_summary)

con.close()


### What just happened?
- The **`JOIN df_customers`** clause reaches into Python's local scope and scans the Pandas DataFrame without copying it into DuckDB storage.
- This pattern is extremely useful for **enriching database tables** with reference data loaded from an API or config file.
- DuckDB's planner will **hash-join** a small Pandas DataFrame against a large persistent table — the Pandas side becomes the build side of the hash join.
- Using an explicit `con` connection (not `duckdb.sql()`) is important here because the `transactions` table lives in that connection's in-memory database.


## Step 6 · Benchmark — DuckDB vs pandas for aggregation

Let's put numbers to the zero-copy claim. We'll generate a 5 million-row dataset and compare:
1. **`duckdb.sql().df()`** — DuckDB reads Pandas in-place, aggregates, returns DataFrame
2. **Pure Pandas `groupby`** — standard Pandas aggregation

> **Note:** Results vary by machine. The key insight is the relative order, not absolute numbers.


In [ ]:
import duckdb
import pandas as pd
import numpy as np
import time

# ── Generate 5M rows ──────────────────────────────────────────────────────────
N = 5_000_000
rng = np.random.default_rng(0)

print(f"Generating {N:,} rows...")
gen_start = time.perf_counter()
df_big = pd.DataFrame({
    'category': rng.choice(['A', 'B', 'C', 'D', 'E'], N),
    'region':   rng.choice(['North', 'South', 'East', 'West'], N),
    'value':    rng.uniform(1, 1000, N),
    'qty':      rng.integers(1, 100, N),
})
gen_ms = (time.perf_counter() - gen_start) * 1000
print(f"Generated in {gen_ms:.0f} ms — shape: {df_big.shape}")
print(f"Memory: {df_big.memory_usage(deep=True).sum() / 1e6:.1f} MB\n")

# ── Benchmark 1: DuckDB ───────────────────────────────────────────────────────
SQL = """
    SELECT category,
           region,
           COUNT(*)             AS rows,
           ROUND(SUM(value), 2) AS total_value,
           ROUND(AVG(qty), 2)   AS avg_qty
    FROM df_big
    GROUP BY category, region
    ORDER BY category, region
"""

t0 = time.perf_counter()
result_duck = duckdb.sql(SQL).df()
duck_ms = (time.perf_counter() - t0) * 1000

# ── Benchmark 2: Pure Pandas ──────────────────────────────────────────────────
t0 = time.perf_counter()
result_pd = (
    df_big.groupby(['category', 'region'])
    .agg(
        rows=('value', 'count'),
        total_value=('value', 'sum'),
        avg_qty=('qty', 'mean')
    )
    .reset_index()
    .sort_values(['category', 'region'])
)
pd_ms = (time.perf_counter() - t0) * 1000

# ── Results ───────────────────────────────────────────────────────────────────
print(f"DuckDB:      {duck_ms:6.0f} ms")
print(f"Pandas:      {pd_ms:6.0f} ms")
print(f"Speedup:     {pd_ms / duck_ms:.1f}x  (DuckDB vs Pandas)")
print(f"\nResult rows: {len(result_duck)} (DuckDB) vs {len(result_pd)} (Pandas)")
print("\nDuckDB result sample:")
print(result_duck.head(4))


### What just happened?
- **DuckDB is faster** for this kind of grouped aggregation because it uses SIMD instructions and vectorised execution on columnar data.
- **Pandas `groupby`** operates row-by-row in its inner loop (despite being written in C); DuckDB processes full batches of 2048 values at once.
- Memory usage is **comparable** — DuckDB reads the same Pandas buffer, so no extra copy is made.
- The speedup grows with row count and aggregation complexity; for simple scalar operations on small DataFrames, Pandas may be faster due to lower setup overhead.


In [ ]:
# Challenge: Multi-library pipeline
#
# Complete the function below:
#   1. Accept a Pandas DataFrame with columns [user_id, product, amount, date]
#   2. Use duckdb.sql() to find the top-N users by total_amount (with a count of orders)
#   3. Return the result as a Polars DataFrame using .pl()
#   4. Then filter the Polars result to keep only users with order_count >= min_orders
#
# Hint: .pl() on a DuckDB relation returns polars.DataFrame
# Hint: use pl.col() for the Polars filter step

import duckdb
import pandas as pd
import polars as pl
import numpy as np

def top_users(
    df: pd.DataFrame,
    n: int = 5,
    min_orders: int = 2
) -> pl.DataFrame:
    # Your solution here
    pass


# Uncomment to test:
# rng = np.random.default_rng(99)
# df_test = pd.DataFrame({
#     'user_id': [f'u{i%8+1:02d}' for i in range(40)],
#     'product': rng.choice(['X', 'Y', 'Z'], 40).tolist(),
#     'amount':  rng.uniform(10, 200, 40).round(2).tolist(),
#     'date':    pd.date_range('2024-01-01', periods=40, freq='D').tolist(),
# })
# print(top_users(df_test, n=5, min_orders=3))


---
## Day 5 key concepts recap

| Concept | What to remember |
|---|---|
| `FROM df` in SQL | DuckDB resolves Python variable names in caller scope — no `register()` needed for `duckdb.sql()` |
| `.df()` / `.fetchdf()` | Converts DuckDB result to `pandas.DataFrame`; column names from SQL aliases |
| Polars integration | Works via Arrow IPC; use `.pl()` to get a `polars.DataFrame` back; collect LazyFrames first |
| `.arrow()` / `.fetch_arrow_table()` | Returns `pyarrow.Table` — zero-copy when type is natively supported |
| Joining mixed backends | DuckDB can JOIN a persistent table with a Pandas DataFrame in one query |
| Benchmark takeaway | DuckDB is typically 3–20× faster than Pandas `groupby` on large aggregations via SIMD vectorisation |

> **Tip:** When DuckDB queries a Pandas or Polars DataFrame it reads the underlying memory buffer directly — no serialisation, no copy. For large frames this can be 10–50× faster than converting first.

---
## What's next
**Day 6** → DuckDB Extensions — httpfs (S3/remote files), spatial, json, and iceberg.

Mark Day 5 complete in your [tracker](../index.html).
